In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import h5py
import os
import subprocess

# --- Paths ---
base_dir = "C:/Users/meggn/Documents/DSC 550 Final Project"
v1_path = f"{base_dir}/Modified Data/singles_V1.csv"
l1_path = f"{base_dir}/Modified Data/singles_L1.csv"
h1_path = f"{base_dir}/Modified Data/singles_H1.csv"
template_path = f"{base_dir}/Modified Data/template_bank.hdf5"
plots_dir = f"{base_dir}/V1 - L1 - H1 Cluster Analysis/Plots"
frames_dir = f"{base_dir}/V1 - L1 - H1 Cluster Analysis/Frames"
ffmpeg_path = f"{base_dir}/ffmpeg-7.1.1-essentials_build/bin/ffmpeg.exe"
output_video_path = f"{base_dir}/V1 - L1 - H1 Cluster Analysis/Virgo - Livingston - Hanford Cluster Evolution.mp4"

# --- Load data ---
v1_df = pd.read_csv(v1_path)
l1_df = pd.read_csv(l1_path)
h1_df = pd.read_csv(h1_path)
print("📄 CSV files loaded.")

# Clip for safety
for df in [v1_df, l1_df, h1_df]:
    df["Mass_1_mf"] = df["Mass_1_mf"].clip(lower=1e-3)
    df["Mass_2_mf"] = df["Mass_2_mf"].clip(lower=1e-3)
    df["SNR_mf"] = df["SNR_mf"].clip(upper=60)
print("🧮 Columns clipped for log scale and SNR max.")

# --- Template bank ---
with h5py.File(template_path, "r") as f:
    mass1, mass2 = [], []
    for key in f.keys():
        try:
            entry = np.array(f[key])
            mass1.append(entry[0][0])
            mass2.append(entry[0][1])
        except:
            continue
template_df = pd.DataFrame({"Mass1": mass1, "Mass2": mass2})
print("📦 Template bank loaded.")

# --- Find common clusters in all 3 detectors ---
common_clusters = sorted(set(v1_df["Cluster ID"])
                         & set(l1_df["Cluster ID"])
                         & set(h1_df["Cluster ID"]))
cluster_counts = v1_df[v1_df["Cluster ID"].isin(common_clusters)] \
                    .groupby("Cluster ID").size().reset_index(name="count") \
                    .sort_values(by="count").reset_index(drop=True)
print(f"🔍 Found {len(cluster_counts)} common clusters across V1, L1, H1.")

# --- Detect resume mode ---
expected_files = len(cluster_counts)
existing_pretty = len(os.listdir(plots_dir)) if os.path.exists(plots_dir) else 0
existing_frames = len(os.listdir(frames_dir)) if os.path.exists(frames_dir) else 0

resume_mode = True
if existing_pretty == expected_files and existing_frames == expected_files:
    print("🧹 All frames present — starting fresh.")
    for folder in [plots_dir, frames_dir]:
        for file in os.listdir(folder):
            os.remove(os.path.join(folder, file))
    resume_mode = False
else:
    print("⏯️ Resume mode: Will skip already completed frames.")
    os.makedirs(plots_dir, exist_ok=True)
    os.makedirs(frames_dir, exist_ok=True)

# --- Ticks ---
m1_ticks = [30, 50, 70, 100, 150, 200, 300, 500]
m2_ticks = [5, 10, 15, 25, 50, 100, 200, 300]

# --- Plotting ---
print("🎨 Generating triple detector plots...")
for i, row in cluster_counts.iterrows():
    cluster_id = row["Cluster ID"]
    plot_path = os.path.join(plots_dir, f"Frame {i+1} - Cluster ID {int(cluster_id)}.png")
    frame_path = os.path.join(frames_dir, f"frame{i+1:05d}.png")

    if resume_mode and os.path.exists(plot_path) and os.path.exists(frame_path):
        print(f"⏭️ Skipping Frame {i+1} (already exists)")
        continue

    v1 = v1_df[v1_df["Cluster ID"] == cluster_id]
    l1 = l1_df[l1_df["Cluster ID"] == cluster_id]
    h1 = h1_df[h1_df["Cluster ID"] == cluster_id]
    total = len(v1) + len(l1) + len(h1)
    
    fig, axs = plt.subplots(1, 3, figsize=(18, 5), sharex=True, sharey=True,
                            gridspec_kw={'wspace': 0.17, 'right': 0.87})

    # --- Template background ---
    for ax in axs:
        ax.scatter(template_df["Mass1"], template_df["Mass2"], s=5, color="lightgray", alpha=0.3, label="Template Bank")
        ax.set_xscale("log")
        ax.set_yscale("log")
        ax.set_xticks(m1_ticks)
        ax.set_xticklabels(m1_ticks)
        ax.set_yticks(m2_ticks)
        ax.set_yticklabels(m2_ticks)
        ax.xaxis.set_major_formatter(ticker.ScalarFormatter())
        ax.yaxis.set_major_formatter(ticker.ScalarFormatter())
        ax.set_xlabel(r"$\log(m_1)$", fontsize=13)
        ax.set_ylabel(r"$\log(m_2)$", fontsize=13)
        ax.legend(loc='upper left', prop={'size': 10})

    # --- Virgo plot ---
    axs[0].scatter(v1["Mass_1_mf"], v1["Mass_2_mf"], c=v1["SNR_mf"], cmap="plasma_r", vmin=0, vmax=60,
                   s=15, edgecolor="k", linewidth=0.2)
    axs[0].set_title(f"Virgo (V1) – {len(v1)}", fontsize=13)

    # --- Livingston plot ---
    axs[1].scatter(l1["Mass_1_mf"], l1["Mass_2_mf"], c=l1["SNR_mf"], cmap="plasma_r", vmin=0, vmax=60,
                   s=15, edgecolor="k", linewidth=0.2)
    axs[1].set_title(f"Livingston (L1) – {len(l1)}", fontsize=13)

    # --- Hanford plot ---
    sc3 = axs[2].scatter(h1["Mass_1_mf"], h1["Mass_2_mf"], c=h1["SNR_mf"], cmap="plasma_r", vmin=0, vmax=60,
                         s=15, edgecolor="k", linewidth=0.2)
    axs[2].set_title(f"Hanford (H1) – {len(h1)}", fontsize=13)
    axs[2].tick_params(labelleft=True)

    # --- Titles ---
    fig.subplots_adjust(top=0.88)
    plt.suptitle("Virgo – Livingston – Hanford Cluster Analysis", fontsize=18, y=1.05)
    fig.text(0.5, 0.95, f"Cluster ID {int(cluster_id)} – Total Triggers: {total}", ha="center", fontsize=14)

    # --- Colorbar ---
    cbar_ax = fig.add_axes([0.89, 0.18, 0.015, 0.65])
    cbar = fig.colorbar(sc3, cax=cbar_ax)
    cbar.set_label(r"SNR ($\rho$)", fontsize=12)

    # --- Save plots ---
    plt.tight_layout(rect=[0, 0, 0.87, 0.90])
    plt.savefig(plot_path, dpi=300, bbox_inches='tight')
    plt.savefig(frame_path, dpi=300, bbox_inches='tight')
    plt.close()

print("✅ All triple-detector plots saved.")

# --- Create video only if all frames are complete ---
num_frames = len([f for f in os.listdir(frames_dir) if f.endswith(".png")])
if num_frames == expected_files:
    print("🎬 Running ffmpeg to create movie...")
    ffmpeg_cmd = [
        ffmpeg_path,
        '-y',
        '-framerate', '20',
        '-i', f'{frames_dir}/frame%05d.png',
        '-vf', 'pad=ceil(iw/2)*2:ceil(ih/2)*2',
        '-c:v', 'libx264',
        '-pix_fmt', 'yuv420p',
        output_video_path
    ]
    subprocess.run(ffmpeg_cmd, check=True)
    print(f"🎥 Video created successfully.")
else:
    print(f"⚠️ Skipped movie creation – only {num_frames}/{expected_files} frames exist.")

